**Stage 3: Window-Level Consensus**

Stable heads use majority vote across the 5 row labels. Considering worst case heads as `Latency` and `Realiability` for controlling semantic communication

In [1]:
import json
import numpy as np
import pandas as pd

RAW_FEATURES = [
    "time", "phy_mcs", "mac_dl_cqi", "mac_dl_ri", "mac_dl_pmi",
    "mac_ul_buffer", "mac_n_prb", "rsrq", "rsrp", "rssi",
    "dl_sinr", "se", "dl_bler", "delay"
]

LABEL_HEADS = [
    "link_quality", "congestion", "latency",
    "interference", "reliability", "scheduler"
]

windowed_output = pd.read_csv("windowed_output_semantic.csv")

MAJORITY_HEADS = ["link_quality", "congestion", "interference", "scheduler"]
WORST_CASE_HEADS = ["latency", "reliability"]
SEVERITY_ORDER = {
    "latency": ["Unknown", "Realtime", "Normal", "Risk", "Critical"],
    "reliability": ["Reliable", "Warning", "Critical"],
}

def majority_vote(values):
    counts = pd.Series(values).value_counts(sort=False)
    max_count = counts.max()
    tied_labels = set(counts[counts == max_count].index)
    for value in values:
        if value in tied_labels:
            return value
    raise ValueError("Unable to determine majority vote.")

def worst_case(head, values):
    rank = {label: index for index, label in enumerate(SEVERITY_ORDER[head])}
    known = [value for value in values if value in rank]
    if not known:
        return "Unknown"
    return max(known, key=lambda value: rank[value])

consensus_rows = []
for _, row in windowed_output.iterrows():
    sample = {
        "window_id": int(row["window_id"]),
        "start_row": int(row["start_row"]),
        "end_row": int(row["end_row"]),
        "start_time": row["time_0"],
        "end_time": row["time_4"],
    }
    for head in MAJORITY_HEADS:
        sample[head] = majority_vote([row[f"{head}_{i}"] for i in range(5)])
    for head in WORST_CASE_HEADS:
        sample[head] = worst_case(head, [row[f"{head}_{i}"] for i in range(5)])
    consensus_rows.append(sample)

window_consensus_labels = pd.DataFrame(consensus_rows)
window_consensus_labels.to_csv("window_consensus_labels.csv", index=False)

print("window_consensus_labels:", window_consensus_labels.shape)
display(window_consensus_labels.head())


window_consensus_labels: (5995, 11)


,window_id,start_row,end_row,start_time,end_time,link_quality,congestion,interference,scheduler,latency,reliability
0,0,0,4,1.743169e+12,1.743169e+12,Excellent,Underutilized,Low,Excellent,Unknown,Reliable
1,1,1,5,1.743169e+12,1.743169e+12,Excellent,Underutilized,Low,Excellent,Unknown,Reliable
2,2,2,6,1.743169e+12,1.743169e+12,Excellent,Underutilized,Low,Excellent,Unknown,Reliable
3,3,3,7,1.743169e+12,1.743169e+12,Excellent,Underutilized,Low,Excellent,Unknown,Reliable
4,4,4,8,1.743169e+12,1.743169e+12,Excellent,Underutilized,Low,Excellent,Unknown,Reliable
